In [32]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [4]:
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
import requests
from langchain_community.tools import DuckDuckGoSearchRun

In [11]:
# Step 1: Create a tool
search_tool = DuckDuckGoSearchRun()
results = search_tool.invoke("Top news in Pune")

In [35]:
@tool
def get_weather_data(city: str) -> str:
    """Get the current weather data for a given city."""
    api_key = os.getenv("WEATHERSTACK_API_KEY")
    url = f"http://api.weatherstack.com/current?access_key={api_key}&query={city}"
    response = requests.get(url)
    data = response.json()
    return data

In [33]:
get_weather_data({"city": "Pune"})

{'request': {'type': 'City',
  'query': 'Pune, India',
  'language': 'en',
  'unit': 'm'},
 'location': {'name': 'Pune',
  'country': 'India',
  'region': 'Maharashtra',
  'lat': '18.533',
  'lon': '73.867',
  'timezone_id': 'Asia/Kolkata',
  'localtime': '2025-12-28 00:24',
  'localtime_epoch': 1766881440,
  'utc_offset': '5.50'},
 'current': {'observation_time': '06:54 PM',
  'temperature': 17,
  'weather_code': 113,
  'weather_icons': ['https://cdn.worldweatheronline.com/images/wsymbols01_png_64/wsymbol_0008_clear_sky_night.png'],
  'weather_descriptions': ['Clear '],
  'astro': {'sunrise': '07:05 AM',
   'sunset': '06:07 PM',
   'moonrise': '12:41 PM',
   'moonset': '12:33 AM',
   'moon_phase': 'Waxing Gibbous',
   'moon_illumination': 52},
  'air_quality': {'co': '695.85',
   'no2': '32.85',
   'o3': '86',
   'so2': '13.45',
   'pm2_5': '43.65',
   'pm10': '44.85',
   'us-epa-index': '3',
   'gb-defra-index': '3'},
  'wind_speed': 4,
  'wind_degree': 179,
  'wind_dir': 'S',
  'pre

In [8]:
results

"Marathi Pune News Read all latest Pune News headlines, breaking news , daily news including crimes, politics and other daily happenings from Pune . Pune News Updates: As rallies protesting the alleged communal attack targeting Muslims in Tripura turned violent in certain regions of Maharashtra ... The Indian Express website has been rated GREEN for its credibility and trustworthiness by Newsguard, a global service that rates news sources for ... Stay updated with the latest news in Pune at Hello Punekars. ... Justice prevails as Pune 's Industries Department Council resolves 75% of MSME ... Bollywood singer Shaan, whose real name is Shantanu Mukherjee, and his wife Radhika Mukherjee have bought a luxury bungalow in Prabhachiwadi, Pune ..."

In [22]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

In [10]:
from langchain_classic.agents import create_react_agent, AgentExecutor
from langchain_classic import hub

In [12]:
# Step 2: Pull the ReAct prompt from LangChain Hub
prompt = hub.pull("hwchase17/react")

In [13]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [36]:
# Step 3: Create a ReAct agent manually with the prompt

agent = create_react_agent(
    llm = llm,
    tools = [search_tool, get_weather_data],
    prompt=prompt
)

In [37]:
# Step 4: Wrap the agent in an AgentExecutor

agent_executor = AgentExecutor(
    agent=agent,
    tools=[search_tool, get_weather_data],
    verbose=True
)

In [25]:
# Step 5: Invoke
response = agent_executor.invoke({"input": "What is the latest news about AI in Pune?"})
print(response)



> Entering new AgentExecutor chain...
I need to find the latest news about AI in Pune. I will use a search engine to look for this information.
Action: duckduckgo_search
Action Input: latest news AI Punepune . raipur. rajkot.Uttar Pradesh: Hauled up for using AI during exam, girl dies by suicide. Tech News . Bill Gates on co-founder Paul Allen: ‘He was a problem’; got him drunk, gave him marijuana. Pune hiring trends: Depth in MLOps, embedded AI , vision, and safety engineering. For dev leaders: a practical playbook inside heavy equipment. Prioritize high-ROI features: Predictive maintenance, safety alerts, and workflow automation usually pay first. The Forest Department believes that this AI -based system will play a significant role in creating a safer environment in rural areas. Officials said that, given the positive results, plans are to implement this system in other sensitive regions as well. Latest News . ' AI ' will explain the pune porsche car accident; will be assisted by 

In [26]:
response['output']

'The latest news about AI in Pune includes a growing demand for expertise in MLOps, embedded AI, vision, and safety engineering. An AI-based system is being implemented to enhance safety in rural areas, with plans for wider adoption. AI will also be utilized to reconstruct and explain the Pune Porsche car accident. Furthermore, a Pune-based startup co-founder views AI as a powerful enabler rather than a replacement for human skills.'

In [38]:
response = agent_executor.invoke({"input": "What is capital of Maharashtra and what is its current weather?"})
print(response)



> Entering new AgentExecutor chain...
I need to find out the capital of Maharashtra and then get the current weather for that capital city. I can use duckduckgo_search for the capital and then get_weather_data for the weather.
Action: duckduckgo_search
Action Input: capital of MaharashtraMumbai (/ mʊmˈbaɪ / muum-BY; Marathi: Mumbaī, pronounced [ˈmumbəi] ⓘ), also known as Bombay (/ bɒmˈbeɪ / bom-BAY; its official name until 1995), is the capital city of the Indian state of Maharashtra . Maharashtra is a state in India. Its capital is Mumbai. Maharashtra is one of India’s biggest commercial and industrial centres, and it has played a significant role in the country’s social and political life. It is a leader among Indian states in terms of agricultural and industrial production, trade and transport, and education. Where is Maharashtra located? Maharashtra occupies a substantial portion of the Deccan plateau in the western peninsular part of the Indian subcontinent. It is bounded by six